# Full reproduction in Google Colab

This notebook reproduces the public ERP-MCDA benchmark from the repository commit specified below. It clones the repository, checks out that commit, verifies `HEAD`, prepares the tested runtime and runs the full reproduction.


In [ ]:
REPOSITORY_URL = "https://github.com/myresearchbvp/ERP-MCDA-Benchmark.git"
EXACT_COMMIT = "cb3c60a3fe4038e8c51c7f65d69f60d63a8ae3dc"

from pathlib import Path
import subprocess, shutil, sys
from google.colab import files

if not REPOSITORY_URL or not EXACT_COMMIT:
    raise ValueError("Set REPOSITORY_URL and EXACT_COMMIT to the real public repository and an exact commit hash.")

base = Path.cwd()
repo = base / "erp_mcda_repository"
runtime_root = base / "erp_mcda_runtime"
if repo.exists(): shutil.rmtree(repo)
if runtime_root.exists(): shutil.rmtree(runtime_root)

def run(cmd, cwd=None):
    print("$", " ".join(str(x) for x in cmd), flush=True)
    cp = subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(cp.stdout, end="", flush=True)
    if cp.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {cp.returncode}: {cmd}")
    return cp.stdout.strip()

run(["git", "clone", REPOSITORY_URL, str(repo)])
run(["git", "checkout", EXACT_COMMIT], cwd=repo)
head = run(["git", "rev-parse", "HEAD"], cwd=repo)
expected = run(["git", "rev-parse", EXACT_COMMIT], cwd=repo)
if head != expected:
    raise RuntimeError(f"HEAD mismatch: expected {expected}, observed {head}")

sys.path.insert(0, str(repo / "src"))
from pipeline.colab_runtime import prepare_runtime, run_full_reproduction, write_runtime_record

python_exe, route, info = prepare_runtime(repo, runtime_root)
print(f"[runtime] Selected route: {route}", flush=True)
print(f"[runtime] Selected environment: {info}", flush=True)
reproduced = repo / "reproduced"
try:
    run_full_reproduction(python_exe, repo, reproduced)
    write_runtime_record(reproduced / "COLAB_RUNTIME_INFO.txt", route=route, info=info)
except Exception as exc:
    print(f"REPRODUCTION_FAILURE: {type(exc).__name__}: {exc}", flush=True)
    raise RuntimeError("Full repository reproduction failed; inspect the streamed diagnostics above.") from exc

archive_base = base / "erp_mcda_full_reproduction_result"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=reproduced))
print(f"[result] Full reproduction PASS. Packaging {archive_path.name}", flush=True)
files.download(str(archive_path))
